# Global and Local Interpretability for a Neural Network on the Bank Marketing Dataset

This notebook introduces two broad forms of interpretability for machine learning models:

- **Global interpretability**, which helps us understand the model’s overall behaviour across the dataset
- **Local interpretability**, which helps us explain an individual prediction

We will use the **UCI Bank Marketing** dataset stored in `bank-full.csv`, train a basic **neural network classifier** using **scikit-learn**, and then apply:

- **Permutation Feature Importance** as a global method that ranks features
- **Partial Dependence Plots (PDPs)** as a global interpretability method
- **LIME** as a local interpretability method
- **SHAP** as a local interpretability method


## Learning goals

By the end of this notebook, you should be able to:

- load and prepare the Bank Marketing dataset
- train a basic neural network classifier with scikit-learn
- evaluate the classifier on a validation set
- interpret the model globally using **permutation importance**
- interpret the model globally using **partial dependence plots**
- interpret a single prediction locally using **LIME**
- interpret a single prediction locally using **SHAP**


## Why these methods?

A neural network can produce strong predictive performance, but its internal reasoning is often difficult to inspect directly. A useful way to begin is with **Permutation Feature Importance**, which ranks features by measuring how much model performance falls when each feature is shuffled. We then use **Partial Dependence Plots** to visualise how predictions change on average as one feature varies. For local explanations, **LIME** and **SHAP** focus on the factors behind one specific prediction.


In [ ]:
# Install additional interpretability libraries if needed
%pip install -q lime shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from lime.lime_tabular import LimeTabularExplainer
import shap

## Load the dataset

The file `bank-full.csv` is the larger Bank Marketing dataset from the UCI Machine Learning Repository. The target variable is `y`, which indicates whether the client subscribed to a term deposit.


In [ ]:
df = pd.read_csv("bank-full.csv", sep=";")
df.head()

In [ ]:
df.info()

## Select predictors and target

We will use a mix of categorical and numerical predictors and model the binary target `y`.


In [ ]:
target = "y"

categorical_features = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month", "poutcome"
]

numerical_features = [
    "age", "balance", "day", 
    "campaign", "pdays", "previous"
]

all_features = categorical_features + numerical_features

df = df[all_features + [target]].copy()
df[target] = df[target].map({"yes": 1, "no": 0})

df.head()

## Train-validation split

We will split the data into training and validation sets. Stratification helps preserve the class balance in both sets.


In [ ]:
X = df[all_features]
y = df[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])

## Preprocessing and neural network model

The neural network will be built using **scikit-learn’s `MLPClassifier`**. We will preprocess the data in a pipeline:

- one-hot encode categorical variables
- standardise numerical variables
- fit a basic neural network classifier


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

mlp_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", MLPClassifier(
        hidden_layer_sizes=(30,),
        activation="relu",
        solver="adam",
        max_iter=300,
        random_state=42
    ))
])

## Train the neural network

In [ ]:
mlp_pipeline.fit(X_train, y_train)

## Evaluate the model

Before interpreting the model, it is useful to check that it has learned something meaningful.


In [ ]:
y_pred = mlp_pipeline.predict(X_valid)

acc = accuracy_score(y_valid, y_pred)
misclassification_rate = 1 - acc

print("Validation accuracy:", round(acc, 4))
print("Validation misclassification rate:", round(misclassification_rate, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, y_pred))
print("\nClassification Report:")
print(classification_report(y_valid, y_pred, digits=4))

## Global interpretability: Permutation Feature Importance

**Permutation Feature Importance** is a simple global interpretability method that ranks features by measuring how much the model’s performance decreases when the values of one feature are randomly shuffled. If shuffling a feature causes a large drop in performance, the model is relying more heavily on that feature.

We will calculate permutation importance on the validation set and rank the original input features.


In [ ]:
perm_result = permutation_importance(
    mlp_pipeline,
    X_valid,
    y_valid,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)

perm_importance_df = pd.DataFrame({
    "Feature": all_features,
    "Importance Mean": perm_result.importances_mean,
    "Importance Std": perm_result.importances_std
}).sort_values("Importance Mean", ascending=False)

perm_importance_df

In [ ]:
top_n = 10
top_perm = perm_importance_df.head(top_n).iloc[::-1]

plt.figure(figsize=(8, 5))
plt.barh(top_perm["Feature"], top_perm["Importance Mean"], xerr=top_perm["Importance Std"])
plt.xlabel("Decrease in accuracy after shuffling")
plt.ylabel("Feature")
plt.title("Permutation Feature Importance (Top 10)")
plt.tight_layout()
plt.show()

## Global interpretability: Partial Dependence Plots (PDP)

A **partial dependence plot** shows how the model’s prediction changes, on average, as one feature varies while averaging over the values of the other features.

We will plot PDPs for a few example features. Here we use the top numerical features from the permutation-importance ranking.


In [ ]:
top_numeric_features = [
    f for f in perm_importance_df["Feature"].tolist()
    if f in numerical_features
][:4]

top_numeric_features

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
PartialDependenceDisplay.from_estimator(
    mlp_pipeline,
    X_valid,
    features=top_numeric_features,
    kind="average",
    grid_resolution=30,
    ax=ax
)
plt.tight_layout()
plt.show()

## Optional extension: PDP with ICE curves

ICE curves show how predictions change for individual observations, while PDP shows the average pattern. This gives a richer global view.


In [ ]:
feature_for_ice = top_numeric_features[0]

fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(
    mlp_pipeline,
    X_valid,
    features=[feature_for_ice],
    kind="both",
    subsample=200,
    grid_resolution=30,
    random_state=42,
    ax=ax
)
plt.tight_layout()
plt.show()

## Prepare transformed data for local explanations

LIME and SHAP are easier to apply when we work with the model’s transformed input matrix. We therefore transform the training and validation data using the preprocessing step and extract the processed feature names.


In [ ]:
fitted_preprocessor = mlp_pipeline.named_steps["preprocessor"]
fitted_model = mlp_pipeline.named_steps["model"]

X_train_processed = fitted_preprocessor.transform(X_train)
X_valid_processed = fitted_preprocessor.transform(X_valid)

X_train_processed = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
X_valid_processed = X_valid_processed.toarray() if hasattr(X_valid_processed, "toarray") else X_valid_processed

feature_names_processed = fitted_preprocessor.get_feature_names_out()

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)
print("Number of processed features:", len(feature_names_processed))

## Local interpretability: LIME

LIME explains an individual prediction by perturbing the selected instance locally and fitting a simple interpretable surrogate model nearby.

We will explain one validation instance.


In [ ]:
instance_idx = 0
instance = X_valid_processed[instance_idx]

print("Chosen validation instance index:", instance_idx)
print("True label:", y_valid.iloc[instance_idx])
print("Predicted probabilities:", fitted_model.predict_proba(instance.reshape(1, -1)))

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train_processed,
    feature_names=feature_names_processed,
    class_names=["No", "Yes"],
    mode="classification"
)

lime_exp = lime_explainer.explain_instance(
    data_row=instance,
    predict_fn=fitted_model.predict_proba,
    num_features=10
)

In [ ]:
lime_exp.as_list()

In [ ]:
fig = lime_exp.as_pyplot_figure()
plt.tight_layout()
plt.show()

## Local interpretability: SHAP

SHAP explains predictions using **Shapley values**. For black-box models, `KernelExplainer` is a practical model-agnostic option.

We will explain the same validation instance.


In [ ]:
background_size = 100
background = shap.sample(X_train_processed, background_size, random_state=42)

def predict_positive_class(X):
    return fitted_model.predict_proba(X)[:, 1]

shap_explainer = shap.KernelExplainer(
    model=predict_positive_class,
    data=background
)

In [ ]:
instance_for_shap = X_valid_processed[instance_idx:instance_idx+1]

shap_values = shap_explainer.shap_values(instance_for_shap, nsamples=100)

In [ ]:
if isinstance(shap_values, list):
    shap_values_to_plot = shap_values[0][0]
else:
    shap_values_to_plot = np.array(shap_values)[0]

base_value = shap_explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = np.array(base_value).flatten()[0]

shap_explanation = shap.Explanation(
    values=shap_values_to_plot,
    base_values=base_value,
    data=instance_for_shap[0],
    feature_names=feature_names_processed
)

shap_explanation

In [ ]:
shap.plots.waterfall(shap_explanation, max_display=10)

In [ ]:
shap.plots.bar(shap_explanation, max_display=10)

## Compare the explanation methods

These methods answer different questions:

- **Permutation importance** ranks features globally according to their contribution to predictive performance
- **PDP** shows how the model prediction changes on average as a feature varies
- **LIME** explains one prediction using a local surrogate model
- **SHAP** explains one prediction using Shapley-value-based contributions

When the same features appear prominently across methods, that can increase confidence in the interpretation. Differences can also occur because each method captures a different aspect of model behaviour.


## Questions for discussion

1. Which features were ranked highest by **permutation importance**?
2. Do the top-ranked features also appear important in the **partial dependence plots**?
3. For the chosen case, which features most strongly influenced the prediction according to **LIME**?
4. For the same case, which features most strongly influenced the prediction according to **SHAP**?
5. Do the local and global explanations tell a broadly consistent story?


## Final summary

In this notebook, we trained a basic **scikit-learn neural network** on the **Bank Marketing** dataset and used both **global** and **local** interpretability methods. We used **permutation importance** to rank features globally, **partial dependence plots** to understand the model’s average behaviour, and **LIME** and **SHAP** to explain an individual prediction. Together, these methods provide complementary views of a neural network model that would otherwise be difficult to interpret directly.
